In [13]:
import litellm
import json
from tavily import TavilyClient
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import List, Optional
load_dotenv(override=True)

True

In [17]:
TAVILY_KEY = os.getenv("TAVILY_KEY")
if not TAVILY_KEY:
    raise ValueError("TAVILY_KEY not found in environment variables")

tavily = TavilyClient(api_key=TAVILY_KEY)


In [18]:
#defining tool for searching query through travily
def search_tavily(query: str):
    result = tavily.search(query)
    return json.dumps(result, indent=2)


In [20]:
print(search_tavily("What are the top 5 most popular programming languages in 2024?"))

{
  "query": "What are the top 5 most popular programming languages in 2024?",
  "follow_up_questions": null,
  "answer": null,
  "images": [],
  "results": [
    {
      "url": "https://www.hackerrank.com/blog/most-popular-languages-2024/",
      "title": "The Most Popular Programming Languages of 2024 - HackerRank Blog",
      "content": "Python, a versatile and easy-to-read language, has surged in popularity for web development, data analysis, and artificial intelligence (AI) projects. C++ is a fast and powerful programming language widely used in system software, game development, and high-performance applications. While it comes in second for developer popularity, C++ is the third most in-demand programming language, an 8% drop from 2022. SQL climbed in rank to become the fourth most popular coding language among developers in 2023, though its total usage in programming language tests did decrease slightly during the same period. While JavaScript is primarily a front-end programmi

In [21]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_tavily",
            "description": "Search the web for recent information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query for web search"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

In [22]:
messages = [
    {
        "role": "user",
        "content": "What are the latest developments in OpenAI?"
    }
]

In [27]:
response = completion(
    model="gemini/gemini-3.1-flash-lite",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)


In [77]:
message = response.choices[0].message


In [62]:
#checking if tool was requested by model and executing it
if response.choices[0].finish_reason == "tool_calls":
    print(f'tool is called with tool name = {response.choices[0].message.tool_calls[0].function.name}')   

tool is called with tool name = search_tavily


In [80]:
if response.choices[0].finish_reason == "tool_calls":

    #extract tool call
    tool_call = response.choices[0].message.tool_calls[0]
    tool_name = tool_call.function.name
    tool_args = json.loads(tool_call.function.arguments)
    print(tool_args['query'])

    if tool_name == "search_tavily":
        result = search_tavily(**tool_args)
        print(f'Tool execution result: {result}')

        messages.append(message)
        # Add tool result message
        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_name,
                "content": result
            }
        )
    # -----------------------------
        # Second LLM call
        # -----------------------------

        final_response = completion(
            model="gemini/gemini-3.1-flash-lite",
            messages=messages
        )

        print("\nFINAL ANSWER:\n")
        print(final_response.choices[0].message.content)

else:
    print(response.choices[0].message.content)


    

latest news OpenAI developments October 2024
Tool execution result: {
  "query": "latest news OpenAI developments October 2024",
  "follow_up_questions": null,
  "answer": null,
  "images": [],
  "results": [
    {
      "url": "https://www.linkedin.com/pulse/weeks-latest-generative-ai-updates-october-1-2024-symphonyai-omq0c",
      "title": "This week's latest generative AI updates - October 1, 2024 - LinkedIn",
      "content": "OpenAI Establishes Multi-Agent Research Team for Advanced AI Development: OpenAI is forming a research team focused on multi-agent systems",
      "score": 0.9994253,
      "raw_content": null
    },
    {
      "url": "https://medium.com/nlplanet/weekly-ai-news-october-7th-2024-1b96f913aa6b",
      "title": "Weekly AI News \u2014 October 7th 2024 | by Fabio Chiusano - Medium",
      "content": "OpenAI has launched a new \u201cCanvas\u201d interface for ChatGPT, enhancing user interaction with features like side-by-side text and code",
      "score": 0.999189

/Users/rakshit/auto-analyst/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ChatCompletionMessageToolCall` - serialized value may not be as expected [field_name='tool_calls', input_value={'index': 0, 'provider_sp.../f', 'type': 'function'}, input_type=dict])
  return self.__pydantic_serializer__.to_python(



FINAL ANSWER:

As of October 2024, OpenAI has made significant strides in its funding, product interface, and research initiatives. Here are the key developments:

### 1. Record-Breaking Funding Round
In early October, OpenAI successfully secured a massive **$6.6 billion in new funding**. This investment round has reportedly pushed the company's valuation to $157 billion. While there was significant interest from major tech players, companies like Apple reportedly withdrew from negotiations to participate in this specific round.

### 2. Introduction of "Canvas"
OpenAI launched a new interface feature for ChatGPT called **"Canvas."** This is designed to improve how users interact with the AI for writing and coding projects. Instead of a traditional chat-only window, Canvas opens a separate side-by-side workspace where users can directly edit text or code generated by ChatGPT, allowing for better collaboration and more precise refinements.

### 3. "ChatGPT Search"
OpenAI introduced **"C

In [67]:
tool_args

'{"query": "latest news OpenAI developments October 2024"}'

In [ ]:
# Step 1 — Define the tool
tools = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for current information on a topic",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to look up"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Perform a simple calculation given a mathematical expression",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The mathematical expression to calculate (e.g. '2 + 2 * (3/4)')"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "python_code_executor",
            "description": "Execute Python code and return the result",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "The Python code to execute (e.g. 'import math; math.sqrt(16)')"
                    }
                },
                "required": ["code"]
            }
        }
    }
]

In [117]:
messages = []


In [118]:
# Step 2 — Actually run the tool when Claude asks for it
def run_tool(tool_name : str, tool_args):
    if tool_name == "web_search":
        results = tavily.search(tool_args["query"], max_results=3)
        # Just return the text snippets, not the whole object
        return "\n\n".join([r["content"] for r in results["results"]])
    elif tool_name == "calculator":
        # WARNING: using eval can be dangerous in production code. This is just for demonstration purposes.
        try:
            return str(eval(tool_args["expression"]))
        except Exception as e:
            return f"Error evaluating expression: {e}"
    elif tool_name == "python_code_executor":
        # WARNING: using exec can be dangerous in production code. This is just for demonstration purposes.
        local_vars = {}
        try:
            exec(tool_args["code"], {}, local_vars)
            return str(local_vars.get("result", "No result variable set"))
        except Exception as e:
            return f"Error executing code: {e}"

# Step 3 — The agent loop
def run_agent(question):
    messages.append({"role": "user", "content": question})
    print(f"\nQuestion: {question}\n")

    while True:
        response = completion(
            model="gemini/gemini-3.1-flash-lite",   # swap to gpt-4o or anthropic/claude-sonnet-4-20250514
            messages=messages,
            tools=tools
        )
        # print(response.choices)

        choice = response.choices[0]

        # LLM wants to call a tool
        if choice.finish_reason == "tool_calls":
            tool_call = choice.message.tool_calls[0]
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            print(f"Tool called: {tool_name}")
            # print(f"Query: {tool_args['query']}\n")
            print(f"Args: {tool_args}")

            tool_result = run_tool(tool_name, tool_args)

            # Append assistant's tool call message first
            messages.append(choice.message)
            print(choice.message)

            # Then append the tool result
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_result
            })

        # LLM is done — print final answer
        else:
            print(f"Answer:\n{choice.message.content}")
            break

        print("\n--- New LLM Response ---\n")

# run_agent("What is Vaxcyte VAX-31 and how does it compare to Merck CAPVAXIVE?")

In [119]:
run_agent("What is Merck's current stock price?")



Question: What is Merck's current stock price?

Tool called: web_search
Args: {'query': 'Merck stock price'}
Message(content=None, role='assistant', tool_calls=[ChatCompletionMessageToolCall(index=0, provider_specific_fields={'thought_signature': 'EjQKMgEMOdbHorUXssg4k4COyG4JYsm5Fbv0YU7HdLdcsVlD3JX41T5bdgMWVa/Njeibg9mn'}, function=Function(arguments='{"query": "Merck stock price"}', name='web_search'), id='call_78c56c06bf3b4dbfab6c489ad7f6__thought__EjQKMgEMOdbHorUXssg4k4COyG4JYsm5Fbv0YU7HdLdcsVlD3JX41T5bdgMWVa/Njeibg9mn', type='function')], function_call=None, images=[], thinking_blocks=[], provider_specific_fields={'thought_signatures': ['EjQKMgEMOdbHorUXssg4k4COyG4JYsm5Fbv0YU7HdLdcsVlD3JX41T5bdgMWVa/Njeibg9mn']})

--- New LLM Response ---



/Users/rakshit/auto-analyst/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ChatCompletionMessageToolCall` - serialized value may not be as expected [field_name='tool_calls', input_value={'index': 0, 'provider_sp...mn', 'type': 'function'}, input_type=dict])
  return self.__pydantic_serializer__.to_python(


Answer:
Merck & Co., Inc. (MRK) is currently trading at **$111.35**.

*Note: Stock market data fluctuates throughout the trading day. For the most up-to-the-minute price and detailed market performance, please check a financial news website or your brokerage platform.*


In [120]:
print(messages)

[{'role': 'user', 'content': "What is Merck's current stock price?"}, Message(content=None, role='assistant', tool_calls=[{'index': 0, 'provider_specific_fields': {'thought_signature': 'EjQKMgEMOdbHorUXssg4k4COyG4JYsm5Fbv0YU7HdLdcsVlD3JX41T5bdgMWVa/Njeibg9mn'}, 'function': {'arguments': '{"query": "Merck stock price"}', 'name': 'web_search'}, 'id': 'call_78c56c06bf3b4dbfab6c489ad7f6__thought__EjQKMgEMOdbHorUXssg4k4COyG4JYsm5Fbv0YU7HdLdcsVlD3JX41T5bdgMWVa/Njeibg9mn', 'type': 'function'}], function_call=None, images=[], thinking_blocks=[], provider_specific_fields={'thought_signatures': ['EjQKMgEMOdbHorUXssg4k4COyG4JYsm5Fbv0YU7HdLdcsVlD3JX41T5bdgMWVa/Njeibg9mn']}), {'role': 'tool', 'tool_call_id': 'call_78c56c06bf3b4dbfab6c489ad7f6__thought__EjQKMgEMOdbHorUXssg4k4COyG4JYsm5Fbv0YU7HdLdcsVlD3JX41T5bdgMWVa/Njeibg9mn', 'content': "Merck & Co., Inc.'s stock was trading at $105.20 at the beginning of the year. Since then, MRK stock has increased by 5.8% and is now trading at $111.2550.\n\n# Me

In [121]:
run_agent("What about Pfizer?")  # does it remember context?


Question: What about Pfizer?



/Users/rakshit/auto-analyst/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ChatCompletionMessageToolCall` - serialized value may not be as expected [field_name='tool_calls', input_value={'index': 0, 'provider_sp...mn', 'type': 'function'}, input_type=dict])
  return self.__pydantic_serializer__.to_python(


Tool called: web_search
Args: {'query': 'Pfizer stock price'}
Message(content=None, role='assistant', tool_calls=[ChatCompletionMessageToolCall(index=0, provider_specific_fields={'thought_signature': 'EjQKMgEMOdbH5FUZOVFM/u+6Oee4HruxtOJPgVldRYsNDUOfo/a1O3yiirZ1FTXq4plNzvmk'}, function=Function(arguments='{"query": "Pfizer stock price"}', name='web_search'), id='call_d2c0a774b3e04a898dc00a395e81__thought__EjQKMgEMOdbH5FUZOVFM/u+6Oee4HruxtOJPgVldRYsNDUOfo/a1O3yiirZ1FTXq4plNzvmk', type='function')], function_call=None, images=[], thinking_blocks=[], provider_specific_fields={'thought_signatures': ['EjQKMgEMOdbH5FUZOVFM/u+6Oee4HruxtOJPgVldRYsNDUOfo/a1O3yiirZ1FTXq4plNzvmk']})

--- New LLM Response ---



/Users/rakshit/auto-analyst/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ChatCompletionMessageToolCall` - serialized value may not be as expected [field_name='tool_calls', input_value={'index': 0, 'provider_sp...mk', 'type': 'function'}, input_type=dict])
  return self.__pydantic_serializer__.to_python(


Answer:
Pfizer (PFE) is currently trading at **$26.46**.

*Note: Stock prices change constantly throughout the trading day. The figures provided are based on recent market data.*


In [122]:
run_agent("Which one is higher?")  # comparing?


Question: Which one is higher?

Answer:
Based on the information provided:

*   **Merck (MRK)** is trading at **$111.35**.
*   **Pfizer (PFE)** is trading at **$26.46** (with recent data showing ranges between $25.68 and $26.46).

**Merck's stock price is significantly higher** than Pfizer's.


## planner agent added

In [126]:


# load_dotenv()
# tavily = TavilyClient()
model="gemini/gemini-3.1-flash-lite"
# ─── Pydantic Models ───────────────────────────────────────────────
class SubTask(BaseModel):
    task: str
    tool_name: str
    tool_args: dict

class TaskPlan(BaseModel):
    subtasks: List[SubTask]

# ─── Tools ────────────────────────────────────────────────────────
tools = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for current information on a topic",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a mathematical expression",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression to evaluate"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "code_exec",
            "description": "Execute arbitrary Python code and return the output",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {"type": "string", "description": "Python code to execute"}
                },
                "required": ["code"]
            }
        }
    }
]


In [129]:
# ─── Planner ──────────────────────────────────────────────────────
def run_planner(question: str) -> TaskPlan:
    system_prompt = """You are a planner. Your job is to break a user question into subtasks.
Each subtask must use exactly one of these tools: web_search, calculator, code_exec.

Return ONLY a valid JSON object in this exact format — no prose, no markdown, no backticks:
{
  "subtasks": [
    {
      "task": "plain English description of what this step does",
      "tool_name": "web_search | calculator | code_exec",
      "tool_args": { "query": "..." }
    }
  ]
}

tool_args must match the tool:
- web_search  → { "query": "..." }
- calculator  → { "expression": "..." }
- code_exec   → { "code": "..." }
"""

    response = litellm.completion(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
    )

    raw = response.choices[0].message.content
    parsed = json.loads(raw)
    return TaskPlan(**parsed)  # validates against Pydantic model

# ─── Executor ─────────────────────────────────────────────────────
messages = []

def run_agent(question: str):
    messages.append({"role": "user", "content": question})
    print(f"\nQuestion: {question}\n")

    while True:
        response = litellm.completion(
            model=model,
            messages=messages,
            tools=tools
        )

        choice = response.choices[0]

        if choice.finish_reason == "tool_calls":
            tool_call = choice.message.tool_calls[0]
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            print(f"Tool called: {tool_name}")
            print(f"Args: {tool_args}\n")

            tool_result = run_tool(tool_name, tool_args)

            messages.append({
            "role": "assistant",
            "content": None,
            "tool_calls": choice.message.tool_calls
                            })
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_result
            })

        else:
            print(f"Answer:\n{choice.message.content}")
            messages.append({"role": "assistant", "content": choice.message.content})
            break

# ─── Planner + Executor Pipeline ──────────────────────────────────
def run_pipeline(question: str):
    print("\n========== PLANNER ==========")
    plan = run_planner(question)

    for i, subtask in enumerate(plan.subtasks):
        print(f"\nSubtask {i+1}: {subtask.task}")
        print(f"Tool: {subtask.tool_name} | Args: {subtask.tool_args}")

    print("\n========== EXECUTOR ==========")
    # Feed the full plan as context to the executor
    plan_summary = "\n".join(
        [f"Step {i+1}: {s.task} using {s.tool_name}" for i, s in enumerate(plan.subtasks)]
    )
    enriched_question = f"{question}\n\nHere is your execution plan:\n{plan_summary}"
    run_agent(enriched_question)



In [130]:
# ─── Run ──────────────────────────────────────────────────────────
run_pipeline("What is Merck's current stock price, multiply it by 47 shares, then subtract 12.5% tax on gains assuming I bought at $85?")


========== PLANNER ==========

Subtask 1: Find the current stock price of Merck (MRK)
Tool: web_search | Args: {'query': 'current stock price of Merck MRK'}

Subtask 2: Calculate the total value of 47 shares, the total gain from the purchase price of $85, the tax on that gain, and the final net amount
Tool: code_exec | Args: {'code': "current_price = 127.35 # Placeholder value, will be updated by runtime data\nshares = 47\nbuy_price = 85\ntotal_value = current_price * shares\ntotal_gain = (current_price - buy_price) * shares\ntax = total_gain * 0.125\nnet_profit = total_gain - tax\nprint(f'{total_value=}, {net_profit=}')"}

========== EXECUTOR ==========

Question: What is Merck's current stock price, multiply it by 47 shares, then subtract 12.5% tax on gains assuming I bought at $85?

Here is your execution plan:
Step 1: Find the current stock price of Merck (MRK) using web_search
Step 2: Calculate the total value of 47 shares, the total gain from the purchase price of $85, the tax o

In [140]:
from generate_mmm_output import sample_roi,VENDOR_ROI,build_channel,SPEND_RANGES

In [138]:
sample_roi('doximity', 'oncology')

3.07

In [146]:
np.random.randint(*SPEND_RANGES["doximity"])

1154339

In [5]:
# imports

import os
import logging
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from litellm import completion
from tqdm.notebook import tqdm


/Users/rakshit/auto-analyst/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import json
import os
from chromadb.utils import embedding_functions

In [28]:
# Point to your generated data
DATA_FOLDER = "mmm_dummy_data" #setting the folder name where data is stored

# Load one file first — understand it before looping all 50
sample_file = os.listdir(DATA_FOLDER)[0]
with open(f"{DATA_FOLDER}/{sample_file}") as f:
    data = json.load(f)

print(json.dumps(data,indent=2))

{
  "model_id": "pharma_vaccines_vaxneuvance_2024_16",
  "metadata": {
    "industry": "pharma",
    "sub_vertical": "vaccines",
    "brand": "vaxneuvance",
    "modelling_period": {
      "start": "2022-01",
      "end": "2024-12"
    },
    "reporting_period": {
      "start": "2024-01",
      "end": "2024-12"
    }
  },
  "financials": {
    "nrv": 56220997,
    "pgm": 34704660
  },
  "model_summary": {
    "total_revenue": 56220997,
    "base_contribution_pct": 0.56,
    "incremental_contribution_pct": 0.44,
    "r_squared": 0.9,
    "mape": 0.06
  },
  "channels": [
    {
      "name": "salesforce_calls",
      "category": "hcp",
      "tactics": [
        "primary_care_calls",
        "specialty_calls"
      ],
      "spend": 7569971,
      "calls_delivered": 69296,
      "reach": 17717,
      "frequency": 3.2,
      "transformation": {
        "adstock_decay_rate": 0.51
      },
      "coefficient": 0.0077,
      "roi": 1.86,
      "revenue_contribution": 14080146
    },
    {
 

In [7]:
import chromadb

# Initialize with persistent storage (do this EVERY session)
client = chromadb.PersistentClient(path="./mmm_vectorstore")  # points to your stored data

# Get existing collection (don't use get_or_create if you want to fail loudly)
collection = client.get_collection(name="mmm_outputs")

# Verify it loaded correctly
print(collection.count())  # should show your stored doc count

406


In [17]:
meta = data['metadata']
summary = data['model_summary']
fin = data['financials']

summary_text = (
    f"{meta['brand'].title()} is a {meta['sub_vertical']} brand. "
    f"The MMM model covers {meta['modelling_period']['start']} to {meta['modelling_period']['end']}. "
    f"Total revenue is ${fin['nrv']:,}. "
    f"Base contribution is {summary['base_contribution_pct']*100:.0f}% "
    f"and incremental contribution is {summary['incremental_contribution_pct']*100:.0f}%. "
    f"Model R-squared is {summary['r_squared']} with MAPE of {summary['mape']}."
)

In [22]:
# Pick the first channel from the sample file
channel = data["channels"][0]
meta = data["metadata"]

channel_text = (
    f"{channel['name'].title()} is a {channel['category'].upper()} channel "
    f"for {meta['brand'].title()} ({meta['sub_vertical']}). "
    f"Spend was ${channel['spend']:,} with an ROI of {channel['roi']}x. "
    f"Revenue contribution was ${channel['revenue_contribution']:,}. "
    f"Reach was {channel['reach']:,} with frequency of {channel['frequency']}. "
    f"Adstock decay rate is {channel['transformation']['adstock_decay_rate']}. "
    f"Tactics used: {', '.join(channel['tactics'])}."
)

# Add calls_delivered only if it exists
if channel.get("calls_delivered"):
    channel_text += f" Calls delivered: {channel['calls_delivered']:,}."

print(channel_text)

Salesforce_Calls is a HCP channel for Vaxneuvance (vaccines). Spend was $7,569,971 with an ROI of 1.86x. Revenue contribution was $14,080,146. Reach was 17,717 with frequency of 3.2. Adstock decay rate is 0.51. Tactics used: primary_care_calls, specialty_calls. Calls delivered: 69,296.


In [70]:
def chunk_mmm_file(filepath):
    with open(filepath) as f:
        data = json.load(f)

    meta = data['metadata']
    summary = data['model_summary']
    fin = data['financials']
    brand    = meta["brand"]
    vertical = meta["sub_vertical"]
    year     = meta["modelling_period"]["end"].split("-")[0]
    model_id = data["model_id"]

    chunks = []

    # Summary chunk
    summary_text = (
        f"{brand.title()} is a {vertical} brand. "
        f"Total revenue is ${fin['nrv']:,}. "
        f"The MMM model covers {meta['modelling_period']['start']} to {meta['modelling_period']['end']}. "
        f"Base contribution is {summary['base_contribution_pct']*100:.0f}% "
        f"and incremental contribution is {summary['incremental_contribution_pct']*100:.0f}%. "
        f"Model R-squared is {summary['r_squared']} with MAPE of {summary['mape']}."
    )
    chunks.append({
        "text": summary_text,
        "metadata": {
            "type": "summary",
            "brand": brand,
            "sub_vertical": vertical,
            "year": year,
            "model_id": model_id,
            "channel": "all",
            "category": "all"
        },
        "id": f"{model_id}_summary"
    })

    # Channel chunks
    for ch in data["channels"]:
        channel_text = (
            f"{ch['name'].title()} is a {ch['category'].upper()} channel "
            f"for {brand.title()} ({vertical}). "
            f"Revenue contribution was ${ch['revenue_contribution']:,}. "
            f"Spend was ${ch['spend']:,} with an ROI of {ch['roi']}x. "
            
            f"Reach was {ch['reach']:,} with frequency of {ch['frequency']}. "
            f"Adstock decay rate is {ch['transformation']['adstock_decay_rate']}. "
            f"Tactics used: {', '.join(ch['tactics'])}."
        )
        if ch.get("calls_delivered"):
            channel_text += f" Calls delivered: {ch['calls_delivered']:,}."
        
        chunks.append({
            "text": channel_text,
            "metadata": {
                "type": "channel",
                "brand": brand,
                "sub_vertical": vertical,
                "year": year,
                "model_id": model_id,
                "channel": ch["name"],
                "category": ch["category"]
            },
            "id": f"{model_id}_{ch['name']}"
        })
    
    return chunks



In [71]:
# Test it on one file
chunks = chunk_mmm_file(f"{DATA_FOLDER}/{sample_file}")
print(f"Total chunks from one file: {len(chunks)}")
for c in chunks:
    print(f"\n[{c['metadata']['type'].upper()}] {c['id']}")
    print(c['text'])

Total chunks from one file: 9

[SUMMARY] pharma_vaccines_vaxneuvance_2024_16_summary
Vaxneuvance is a vaccines brand. Total revenue is $56,220,997. The MMM model covers 2022-01 to 2024-12. Base contribution is 56% and incremental contribution is 44%. Model R-squared is 0.9 with MAPE of 0.06.

[CHANNEL] pharma_vaccines_vaxneuvance_2024_16_salesforce_calls
Salesforce_Calls is a HCP channel for Vaxneuvance (vaccines). Revenue contribution was $14,080,146. Spend was $7,569,971 with an ROI of 1.86x. Reach was 17,717 with frequency of 3.2. Adstock decay rate is 0.51. Tactics used: primary_care_calls, specialty_calls. Calls delivered: 69,296.

[CHANNEL] pharma_vaccines_vaxneuvance_2024_16_sfmc
Sfmc is a HCP channel for Vaxneuvance (vaccines). Revenue contribution was $129,686. Spend was $59,489 with an ROI of 2.18x. Reach was 25,503 with frequency of 3.7. Adstock decay rate is 0.48. Tactics used: hq_emails.

[CHANNEL] pharma_vaccines_vaxneuvance_2024_16_pulsepoint
Pulsepoint is a HCP channel 

In [72]:
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

client = chromadb.PersistentClient(path="./mmm_vectorstore")

collection = client.get_or_create_collection(
    name="mmm_outputs",
    embedding_function=embedding_fn
)

print(f"Collection created: {collection.name}")
print(f"Documents in collection: {collection.count()}")

Collection created: mmm_outputs
Documents in collection: 406


In [73]:
# Don't ingest all 50 yet — do one file and query it
chunks = chunk_mmm_file(f"{DATA_FOLDER}/{sample_file}")

collection.add(
    documents=[c["text"] for c in chunks],
    metadatas=[c["metadata"] for c in chunks],
    ids=[c["id"] for c in chunks]
)

print(f"Ingested {len(chunks)} chunks")
print(f"Total in collection now: {collection.count()}")

Ingested 9 chunks
Total in collection now: 406


In [75]:
# This is the most important cell — does retrieval actually work?
results = collection.query(
    query_texts=["what is the revenue of Proquad 2024?"],
    n_results=3
)

for i, doc in enumerate(results["documents"][0]):
    print(f"\n--- Result {i+1} ---")
    print(doc)
    print(f"Metadata: {results['metadatas'][0][i]}")


--- Result 1 ---
Display is a CONSUMER channel for Proquad (vaccines). Spend was $245,750 with an ROI of 1.23x. Revenue contribution was $302,272. Reach was 6,500,976 with frequency of 3.8. Adstock decay rate is 0.24. Tactics used: banners.
Metadata: {'brand': 'proquad', 'model_id': 'pharma_vaccines_proquad_2024_26', 'sub_vertical': 'vaccines', 'type': 'channel', 'category': 'consumer', 'year': '2024', 'channel': 'display'}

--- Result 2 ---
Display is a CONSUMER channel for Proquad (vaccines). Spend was $255,608 with an ROI of 1.3x. Revenue contribution was $332,290. Reach was 6,665,618 with frequency of 3.4. Adstock decay rate is 0.28. Tactics used: banners.
Metadata: {'category': 'consumer', 'brand': 'proquad', 'type': 'channel', 'channel': 'display', 'model_id': 'pharma_vaccines_proquad_2024_18', 'sub_vertical': 'vaccines', 'year': '2024'}

--- Result 3 ---
Proquad is a vaccines brand. The MMM model covers 2022-01 to 2024-12. Total revenue is $110,542,173. Base contribution is 59%

In [76]:
results = collection.query(query_texts=["which channels have highest revenue contribution?"],n_results=3)

In [77]:
results['documents']

[['Streaming_Tv is a CONSUMER channel for Vaxelis (vaccines). Spend was $3,973,124 with an ROI of 2.22x. Revenue contribution was $8,820,335. Reach was 5,775,855 with frequency of 1.6. Adstock decay rate is 0.37. Tactics used: ctv, ott.',
  'Streaming_Tv is a CONSUMER channel for Vaqta (vaccines). Spend was $1,147,511 with an ROI of 3.27x. Revenue contribution was $3,752,360. Reach was 2,276,570 with frequency of 1.6. Adstock decay rate is 0.35. Tactics used: ctv, ott.',
  'Streaming_Tv is a CONSUMER channel for Qliftara (oncology). Spend was $1,235,226 with an ROI of 1.97x. Revenue contribution was $2,433,395. Reach was 2,338,291 with frequency of 2.8. Adstock decay rate is 0.41. Tactics used: ctv, ott.']]

In [3]:
def ingest_all(folder_path):
    all_files = [f for f in os.listdir(folder_path) if f.endswith(".json")]
    
    for i, filename in enumerate(all_files):
        filepath = os.path.join(folder_path, filename)
        chunks = chunk_mmm_file(filepath)
        
        collection.add(
            documents=[c["text"] for c in chunks],
            metadatas=[c["metadata"] for c in chunks],
            ids=[c["id"] for c in chunks]
        )
        print(f"[{i+1}/{len(all_files)}] Ingested {filename} — {len(chunks)} chunks")
    
    print(f"\nDone. Total chunks in collection: {collection.count()}")

ingest_all(DATA_FOLDER)

NameError: name 'DATA_FOLDER' is not defined

In [79]:
result = collection.query(
    query_texts=["total revenue and model performance"],
    n_results=3,
    where={"$and": [{"brand": {"$eq": "proquad"}}, {"year": {"$eq": "2024"}}]}
)

In [80]:
result['documents']

[['Display is a CONSUMER channel for Proquad (vaccines). Spend was $255,608 with an ROI of 1.3x. Revenue contribution was $332,290. Reach was 6,665,618 with frequency of 3.4. Adstock decay rate is 0.28. Tactics used: banners.',
  'Display is a CONSUMER channel for Proquad (vaccines). Spend was $245,750 with an ROI of 1.23x. Revenue contribution was $302,272. Reach was 6,500,976 with frequency of 3.8. Adstock decay rate is 0.24. Tactics used: banners.',
  'Proquad is a vaccines brand. The MMM model covers 2022-01 to 2024-12. Total revenue is $110,542,173. Base contribution is 59% and incremental contribution is 41%. Model R-squared is 0.95 with MAPE of 0.09.']]

In [ ]:
CATEGORIES = []
COLORS = ['cyan', 'blue', 'brown', 'orange', 'yellow', 'green' , 'purple', 'red']

In [98]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser" 
import numpy as np

# ── Build labels and colors ───────────────────────────────────────
# Color by sub_vertical
verticals  = [m["sub_vertical"] for m in metadatas]
brands     = [m["brand"] for m in metadatas]
types      = [m["type"] for m in metadatas]
channels   = [m["channel"] for m in metadatas]
years      = [m["year"] for m in metadatas]

vertical_palette = {
    "oncology": "#e63946",
    "vaccines":  "#2a9d8f",
    "pharma":    "#e9c46a",
}
colors = [vertical_palette.get(v, "#aaa") for v in verticals]

# ── Marker shape by chunk type ────────────────────────────────────
# summary → star, channel → circle
symbols = ["star" if t == "summary" else "circle" for t in types]

# ── Hover text ────────────────────────────────────────────────────
hover_texts = [
    f"<b>{brands[i].title()} — {verticals[i]}</b><br>"
    f"Type: {types[i]}<br>"
    f"Channel: {channels[i]}<br>"
    f"Year: {years[i]}<br>"
    f"<br><i>{documents[i][:120]}...</i>"
    for i in range(len(documents))
]

# ── One trace per vertical (for legend) ──────────────────────────
fig = go.Figure()

for vertical, color in vertical_palette.items():
    mask = [i for i, v in enumerate(verticals) if v == vertical]

    # summary points — stars
    summary_idx = [i for i in mask if types[i] == "summary"]
    fig.add_trace(go.Scatter(
        x=reduced[summary_idx, 0],
        y=reduced[summary_idx, 1],
        mode="markers",
        name=f"{vertical} — summary",
        marker=dict(
            symbol="star",
            size=12,
            color=color,
            opacity=0.9,
            line=dict(width=1, color="white")
        ),
        text=[hover_texts[i] for i in summary_idx],
        hoverinfo="text"
    ))

    # channel points — circles
    channel_idx = [i for i in mask if types[i] == "channel"]
    fig.add_trace(go.Scatter(
        x=reduced[channel_idx, 0],
        y=reduced[channel_idx, 1],
        mode="markers",
        name=f"{vertical} — channel",
        marker=dict(
            symbol="circle",
            size=6,
            color=color,
            opacity=0.55,
            line=dict(width=0.5, color="white")
        ),
        text=[hover_texts[i] for i in channel_idx],
        hoverinfo="text"
    ))

fig.update_layout(
    title=dict(
        text="MMM Vectorstore — t-SNE Visualization<br>"
             "<sup>Stars = summary chunks | Circles = channel chunks | Color = sub-vertical</sup>",
        font=dict(size=16)
    ),
    width=1300,
    height=800,
    plot_bgcolor="#0f172a",
    paper_bgcolor="#0f172a",
    font=dict(color="white"),
    xaxis=dict(title="t-SNE 1", showgrid=False, zeroline=False),
    yaxis=dict(title="t-SNE 2", showgrid=False, zeroline=False),
    legend=dict(
        bgcolor="#1e293b",
        bordercolor="#334155",
        borderwidth=1
    ),
    margin=dict(r=20, b=40, l=40, t=80)
)

fig.show()

['zerbaxia',
 'del-pif',
 'proquad',
 'lynparza',
 'qliftara',
 'vaxneuvance',
 'welireg',
 'vaxelis',
 'januvia',
 'vaqta',
 'rotateq',
 'gardasil',
 'capvaxive',
 'belsomra',
 'keytruda',
 'lenvima',
 'verquovo',
 'dificid',
 'bridion']

In [8]:
import re


In [9]:
merck_brands = []
for meta in collection.get()['metadatas']:
    merck_brands.append(meta['brand'])


merck_channels = set([
    meta['channel'] 
    for meta in collection.get()['metadatas'] 
    if meta['channel'] != "all"
])

list(set(merck_brands))
list(set(merck_channels))

def regex_parser(question: str) -> dict:
    extracted = {}
    question_lower = question.lower()
    brands = set([b for b in merck_brands if re.search(rf'\b{re.escape(b)}\b', question_lower)])
    year   = re.findall(r'\b(202[0-9])\b' , question)
    channel = set([i for i in merck_channels if i in question_lower])
    category = "hcp" if "hcp" in question_lower else ("consumer" if "consumer" in question_lower else None)
    sub_vertical = "oncology" if "oncology" in question_lower else ("vaccines" if "vaccines" in question_lower else ("pharma" if "pharma" in question_lower else None))
    extracted['brands'] = brands
    extracted['year'] = year[0] if year else None
    extracted['channel'] = channel
    extracted['category'] = category
    extracted['sub_vertical'] = sub_vertical
    
    return extracted

In [10]:
print(regex_parser("What is the ROI of Doximity for Proquad in 2024?"))
print(regex_parser("Compare HCP channels for oncology brands"))
print(regex_parser("What is Salesforce calls performance for vaccines in 2023?"))
print(regex_parser("Which consumer channels have the highest ROI?"))

{'brands': {'proquad'}, 'year': '2024', 'channel': {'doximity'}, 'category': None, 'sub_vertical': None}
{'brands': set(), 'year': None, 'channel': set(), 'category': 'hcp', 'sub_vertical': 'oncology'}
{'brands': set(), 'year': '2023', 'channel': set(), 'category': None, 'sub_vertical': 'vaccines'}
{'brands': set(), 'year': None, 'channel': set(), 'category': 'consumer', 'sub_vertical': None}


In [27]:
from datetime import datetime



In [28]:
def llm_parser(question: str) -> dict:
    prompt = f"""
Extract structured filters from this MMM analytics question.
Return ONLY a JSON object with these keys and current year is {datetime.now().year}:':
{{
  "brands": [],          # list of brand names mentioned
  "year": null,          # year as string or null
  "channel": [],         # list of channel/vendor names
  "category": null,      # "hcp" or "consumer" or null
  "sub_vertical": null   # "oncology" or "vaccines" or "pharma" or null
}}

Known brands: {list(merck_brands)}
Known channels: {list(merck_channels)}

Question: {question}

Return only valid JSON. No explanation.
"""
    response = completion(
        model="gemini/gemini-3.1-flash-lite",
        messages=[{"role": "user", "content": prompt}]
    )

    return json.loads(response.choices[0].message.content)


In [29]:
print(llm_parser("What is the ROI of Doximity for Proquad in 2024?"))

{'brands': ['proquad'], 'year': '2024', 'channel': ['doximity'], 'category': None, 'sub_vertical': None}


In [30]:
def parse_query(question: str) -> dict:
    extracted = regex_parser(question)
    
    # check if regex found anything useful
    has_something = any([
        extracted["brands"],
        extracted["year"],
        extracted["channel"],
        extracted["category"],
        extracted["sub_vertical"]
    ])
    
    if not has_something:
        print("Regex found nothing — falling back to LLM parser")
        extracted = llm_parser(question)
    
    return extracted

In [31]:
parse_query("Which brand had the worst model fit last year?")

Regex found nothing — falling back to LLM parser


{'brands': [],
 'year': '2025',
 'channel': [],
 'category': None,
 'sub_vertical': None}

In [ ]:
where = {'brands': ['Capvaxive','Proquad'],
 'year': '2025',
 'channel': [],
 'category': None,
 'sub_vertical': None}
# where={
#     "$or": [
#         {"source": {"$eq": "veeva"}},
#         {"source": {"$eq": "mvcc"}}
#     ]
# }


['Capvaxive', 'Proquad']

In [48]:
def build_where_clause(filters: dict) -> dict:
    conditions = []

    if filters.get("brands"):
        brands_list = list(filters["brands"])
        if len(brands_list) == 1:
            conditions.append({"brand": {"$eq": brands_list[0]}})
        else:
            conditions.append({"brand": {"$in": brands_list}})

    if filters.get("year"):
        conditions.append({"year": {"$eq": filters["year"]}})

    if filters.get("channel"):
        channels_list = list(filters["channel"])
        if len(channels_list) == 1:
            conditions.append({"channel": {"$eq": channels_list[0]}})
        else:
            conditions.append({"channel": {"$in": channels_list}})

    if filters.get("category"):
        conditions.append({"category": {"$eq": filters["category"]}})

    if filters.get("sub_vertical"):
        conditions.append({"sub_vertical": {"$eq": filters["sub_vertical"]}})

    # wrap correctly based on how many conditions
    if len(conditions) == 0:
        return {}
    elif len(conditions) == 1:
        return conditions[0]
    else:
        return {"$and": conditions}

In [49]:
build_where_clause(where)

{'$and': [{'brand': {'$in': ['Capvaxive', 'Proquad']}},
  {'year': {'$eq': '2025'}}]}

In [55]:
def mmm_retriever(question: str, collection, n_results: int = 10) -> str:
    # 1. parse the question
    filters = parse_query(question)

    where = build_where_clause(filters)
    
    # 3. query ChromaDB
    #    if where is empty dict — query without metadata filter
    #    if where has filters — query with where
    if where:
        results = collection.query(
            query_texts=[question],
            n_results=n_results,
            where=where
        )
    else:
        results = collection.query(
            query_texts=[question],
            n_results=n_results
        )
    
    # 4. format results as plain text string
    #    return something the LLM can read
    #    include the chunk text + metadata for each result
    formatted_results = ""
    for i in range(len(results["documents"][0])):
        doc = results["documents"][0][i]
        meta = results["metadatas"][0][i]
        formatted_results += f"--- Result {i+1} ---\n"
        formatted_results += f"Brand: {meta['brand']}\n"
        formatted_results += f"Year: {meta['year']}\n"
        formatted_results += f"Channel: {meta['channel']}\n"
        formatted_results += f"Category: {meta['category']}\n"
        formatted_results += f"Sub-vertical: {meta['sub_vertical']}\n"
        formatted_results += f"Type: {meta['type']}\n"
        formatted_results += f"Content: {doc}\n\n"
    
    return formatted_results

In [56]:
# print(mmm_retriever("What is the ROI of Doximity for Proquad in 2024?", collection))
print(mmm_retriever("Compare HCP channels for oncology brands", collection))

--- Result 1 ---
Brand: lenvima
Year: 2023
Channel: sfmc
Category: hcp
Sub-vertical: oncology
Type: channel
Content: Sfmc is a HCP channel for Lenvima (oncology). Spend was $217,002 with an ROI of 1.83x. Revenue contribution was $397,113. Reach was 14,353 with frequency of 3.9. Adstock decay rate is 0.44. Tactics used: hq_emails.

--- Result 2 ---
Brand: welireg
Year: 2024
Channel: sfmc
Category: hcp
Sub-vertical: oncology
Type: channel
Content: Sfmc is a HCP channel for Welireg (oncology). Spend was $92,932 with an ROI of 2.1x. Revenue contribution was $195,157. Reach was 8,792 with frequency of 4.0. Adstock decay rate is 0.42. Tactics used: hq_emails.

--- Result 3 ---
Brand: lynparza
Year: 2025
Channel: sfmc
Category: hcp
Sub-vertical: oncology
Type: channel
Content: Sfmc is a HCP channel for Lynparza (oncology). Spend was $212,614 with an ROI of 1.95x. Revenue contribution was $414,597. Reach was 21,949 with frequency of 2.2. Adstock decay rate is 0.35. Tactics used: hq_emails.

--